# Cluster GRASP pocket scores and report pocket residues

This notebook reads per-atom GRASP scores from the PDB **B-factor column**, clusters atoms with any clustering method provided by `site_metrics.py`, and reports residue IDs for every retained pocket. It can run locally or in Google Colab. Use the controls at the bottom and click **Run clustering**; results are also written as CSV files.

## 0. Install dependencies (run this first in Colab)

This installs every direct notebook dependency and the scientific packages imported by `site_metrics.py`. Colab may already provide some of them; `%pip` will reuse compatible installed versions.

In [1]:
%pip install -q numpy pandas biopython scikit-learn scipy MDAnalysis networkx tqdm joblib ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.0 MB/s eta 0:00:00


In [2]:
# A Colab runtime does not automatically contain files beside the notebook.
# If needed, this prompts once to upload site_metrics.py and the scored PDB file.
from pathlib import Path
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

required_example_files = ('site_metrics.py', '6TY3_probs.pdb')
missing_files = [name for name in required_example_files if not Path(name).is_file()]
if IN_COLAB and missing_files:
    print('Upload site_metrics.py and your *_probs.pdb file (the example expects 6TY3_probs.pdb).')
    files.upload()
elif missing_files:
    print('Local note: missing ' + ', '.join(missing_files))
else:
    print('Required example files are available.')

Upload site_metrics.py and your *_probs.pdb file (the example expects 6TY3_probs.pdb).


Saving 6TY3_probs.pdb to 6TY3_probs.pdb
Saving site_metrics.py to site_metrics.py


## 1. Imports

The notebook should be run from this folder so that `site_metrics.py` and the scored PDB are available.

In [3]:
from pathlib import Path
import csv
import numpy as np
import pandas as pd
from Bio.PDB import PDBParser
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets
import site_metrics as grps

## 2. Load coordinates, scores, and residue metadata

Atom order is kept identical across the coordinate, score, and metadata arrays. Residue IDs include chain, residue name, sequence number, and insertion code (when present), for example `X:ARG35` or `A:SER42B`.

In [4]:
def load_scored_pdb(pdb_file):
    pdb_file = Path(pdb_file)
    if not pdb_file.is_file():
        raise FileNotFoundError(f'Cannot find scored PDB: {pdb_file}')

    structure = PDBParser(QUIET=True).get_structure(pdb_file.stem, str(pdb_file))
    model = next(structure.get_models())
    coords, scores, atom_records = [], [], []

    for chain in model:
        chain_id = chain.id.strip() or '_'
        for residue in chain:
            hetero_flag, resseq, insertion_code = residue.id
            insertion_code = insertion_code.strip()
            residue_id = f'{chain_id}:{residue.resname}{resseq}{insertion_code}'
            for atom in residue:
                score = float(atom.get_bfactor())
                coords.append(atom.get_coord())
                scores.append(score)
                atom_records.append({
                    'chain': chain_id, 'resname': residue.resname,
                    'resid': int(resseq), 'insertion_code': insertion_code,
                    'residue_id': residue_id, 'atom_name': atom.name,
                    'hetero_flag': hetero_flag.strip(), 'grasp_score': score,
                })

    all_coords = np.asarray(coords, dtype=float)
    grasp_scores = np.asarray(scores, dtype=float)
    # site_metrics expects the GRASP score in column 1.
    predicted_probs = np.column_stack((1.0 - grasp_scores, grasp_scores))
    return all_coords, predicted_probs, pd.DataFrame(atom_records)

## 3. Cluster, summarize, and export

Pocket ranks below are recomputed from the requested score aggregation so that rank 1 is always the highest-scoring retained pocket. A residue belongs to a pocket when at least one of its atoms passes the GRASP threshold and is assigned to that pocket. DBSCAN noise and atoms rejected by ground-truth association are excluded.

In [5]:
def load_ligand_coordinates(ligand_pdb_file):
    ligand_path = Path(ligand_pdb_file)
    if not ligand_path.is_file():
        raise FileNotFoundError(f'Cannot find ligand PDB: {ligand_path}')
    structure = PDBParser(QUIET=True).get_structure(ligand_path.stem, str(ligand_path))
    model = next(structure.get_models())
    ligand_groups = [np.asarray([atom.get_coord() for atom in residue], dtype=float)
                     for chain in model for residue in chain if len(residue)]
    if not ligand_groups:
        raise ValueError(f'No ligand atoms found in {ligand_path}')
    return ligand_groups


def dispatch_clustering(method, all_coords, predicted_probs, score_threshold, score_type, parameters):
    common = dict(threshold=float(score_threshold), score_type=score_type)
    if method == 'meanshift':
        bandwidth = None if parameters['automatic_bandwidth'] else float(parameters['bandwidth'])
        return grps.cluster_atoms_meanshift(all_coords, predicted_probs, quantile=float(parameters['quantile']), bw=bandwidth, **common)
    if method == 'dbscan':
        return grps.cluster_atoms_DBSCAN(all_coords, predicted_probs, eps=float(parameters['eps']), min_samples=int(parameters['min_samples']), **common)
    if method == 'louvain':
        cutoff = float(parameters['cutoff'])
        adjacency = grps.radius_neighbors_graph(all_coords, radius=cutoff, mode='distance', include_self=False)
        return grps.cluster_atoms_louvain(all_coords, adjacency, predicted_probs, cutoff=cutoff, resolution=float(parameters['resolution']), **common)
    if method in ('single', 'complete'):
        function = getattr(grps, f'cluster_atoms_{method}')
        return function(all_coords, predicted_probs, n_clusters=None, distance_threshold=float(parameters['distance_threshold']), **common)
    if method == 'average':
        return grps.cluster_atoms_average(all_coords, predicted_probs, distance_threshold=float(parameters['distance_threshold']), **common)
    if method == 'ward':
        # Ward modifies its probability input internally, so pass a copy.
        return grps.cluster_atoms_ward(all_coords, predicted_probs.copy(), n_clusters=None, distance_threshold=float(parameters['distance_threshold']), **common)
    if method == 'groundtruth':
        ligand_groups = load_ligand_coordinates(parameters['ligand_pdb_file'])
        return grps.cluster_atoms_groundtruth(all_coords, ligand_groups, predicted_probs, distance_threshold=float(parameters['distance_threshold']), **common)
    raise ValueError(f'Unknown clustering method: {method}')


def run_pocket_clustering(pdb_file, method='average', score_threshold=0.30,
                          min_atom_count=10, score_type='mean', output_prefix=None, **parameters):
    all_coords, predicted_probs, atom_table = load_scored_pdb(pdb_file)
    bind_coords, cluster_ids, all_ids = dispatch_clustering(
        method, all_coords, predicted_probs, score_threshold, score_type, parameters)

    if bind_coords is None:
        empty = pd.DataFrame()
        print(f'No atoms pass the GRASP cutoff {score_threshold:.2f}. Try a lower cutoff.')
        return empty, empty

    atom_table = atom_table.copy()
    atom_table['cluster_id'] = np.asarray(all_ids, dtype=int)
    assigned = atom_table.loc[atom_table['cluster_id'] >= 0].copy()
    passed_count = int(np.sum(predicted_probs[:, 1] >= score_threshold)) if method == 'ward' else int(np.sum(predicted_probs[:, 1] > score_threshold))

    groups = []
    for cluster_id, atoms in assigned.groupby('cluster_id'):
        if len(atoms) < int(min_atom_count):
            continue
        score = atoms['grasp_score'].agg(score_type) if score_type in ('mean', 'sum') else (atoms['grasp_score'] ** 2).sum()
        groups.append((cluster_id, atoms, float(score)))

    groups.sort(key=lambda item: item[2], reverse=True)
    pocket_rows, residue_rows = [], []
    for rank, (cluster_id, atoms, pocket_score) in enumerate(groups, start=1):
        residue_stats = (atoms.groupby(['chain', 'resname', 'resid', 'insertion_code', 'residue_id'], dropna=False)
                         .agg(selected_atom_count=('atom_name', 'size'), max_grasp_score=('grasp_score', 'max'), mean_grasp_score=('grasp_score', 'mean'))
                         .reset_index().sort_values(['chain', 'resid', 'insertion_code']))
        residue_ids = residue_stats['residue_id'].tolist()
        pocket_rows.append({'pocket_rank': rank, 'clustering_method': method, 'pocket_score': pocket_score,
                            'selected_atom_count': len(atoms), 'residue_count': len(residue_ids),
                            'residue_ids': ';'.join(residue_ids)})
        residue_stats.insert(0, 'pocket_rank', rank)
        residue_stats.insert(1, 'clustering_method', method)
        residue_rows.extend(residue_stats.to_dict('records'))

    pockets = pd.DataFrame(pocket_rows)
    residues = pd.DataFrame(residue_rows)
    if output_prefix is None:
        output_prefix = str(Path(pdb_file).with_suffix('')) + '_grasp_pockets'
    output_prefix = Path(output_prefix)
    output_prefix.parent.mkdir(parents=True, exist_ok=True)
    pockets.to_csv(f'{output_prefix}.csv', index=False, float_format='%.4f')
    residues.to_csv(f'{output_prefix}_residues.csv', index=False, float_format='%.4f')

    print(f'{passed_count:,} atoms passed the score cutoff; {len(assigned):,} were assigned; retained {len(pockets)} pockets.')
    print(f'Wrote {output_prefix}.csv and {output_prefix}_residues.csv')
    return pockets, residues

## 4. Friendly controls

- **Clustering method** exposes every `cluster_atoms_*` method in `site_metrics.py`. Its method-specific controls appear automatically.
- **GRASP score cutoff** selects candidate atoms. Start near 0.30; raise it for more confident/smaller pockets.
- **Minimum selected atoms** removes tiny clusters after clustering.
- **Ground truth / ligand association** is an analysis mode and requires a separate ligand PDB. Each residue in that file is treated as one ligand group.

The output prefix may include a folder (for example `results/6TY3`).

In [6]:
control_style = {'description_width': '165px'}
control_layout = widgets.Layout(width='560px')
def float_slider(description, value, minimum, maximum, step):
    return widgets.FloatSlider(value=value, min=minimum, max=maximum, step=step, description=description,
                               continuous_update=False, style=control_style, layout=control_layout)

pdb_file_control = widgets.Text(value='6TY3_probs.pdb', description='Scored PDB:', style=control_style, layout=control_layout)
method_control = widgets.Dropdown(options=[('Mean shift', 'meanshift'), ('DBSCAN', 'dbscan'),
    ('Louvain communities', 'louvain'), ('Single linkage', 'single'), ('Complete linkage', 'complete'),
    ('Average linkage', 'average'), ('Ward linkage', 'ward'), ('Ground truth / ligand', 'groundtruth')],
    value='average', description='Clustering method:', style=control_style, layout=control_layout)
score_control = float_slider('GRASP score cutoff:', 0.30, 0.0, 1.0, 0.01)
min_atoms_control = widgets.IntSlider(value=10, min=1, max=100, description='Minimum atoms:', continuous_update=False, style=control_style, layout=control_layout)
score_type_control = widgets.Dropdown(options=[('Mean atom score', 'mean'), ('Sum of scores', 'sum'), ('Sum of squared scores', 'square')], value='mean', description='Pocket ranking:', style=control_style, layout=control_layout)
prefix_control = widgets.Text(value='6TY3_grasp_pockets', description='Output prefix:', style=control_style, layout=control_layout)

meanshift_auto = widgets.Checkbox(value=True, description='Estimate bandwidth automatically', indent=False)
meanshift_quantile = float_slider('Bandwidth quantile:', 0.30, 0.01, 1.0, 0.01)
meanshift_bandwidth = float_slider('Manual bandwidth (Å):', 5.0, 0.1, 40.0, 0.1)
dbscan_eps = float_slider('Neighbor radius ε (Å):', 3.0, 0.1, 30.0, 0.1)
dbscan_min_samples = widgets.IntSlider(value=5, min=1, max=50, description='Minimum neighbors:', continuous_update=False, style=control_style, layout=control_layout)
louvain_cutoff = float_slider('Graph cutoff (Å):', 5.0, 0.5, 30.0, 0.5)
louvain_resolution = float_slider('Louvain resolution:', 0.05, 0.01, 2.0, 0.01)
linkage_distance = float_slider('Linkage distance (Å):', 20.0, 0.5, 40.0, 0.5)
ligand_file_control = widgets.Text(value='ligand.pdb', description='Ligand PDB:', style=control_style, layout=control_layout)
ligand_distance = float_slider('Association distance (Å):', 5.0, 0.5, 30.0, 0.5)

method_panels = {
    'meanshift': widgets.VBox([meanshift_auto, meanshift_quantile, meanshift_bandwidth]),
    'dbscan': widgets.VBox([dbscan_eps, dbscan_min_samples]),
    'louvain': widgets.VBox([louvain_cutoff, louvain_resolution]),
    'single': widgets.VBox([linkage_distance]), 'complete': widgets.VBox([linkage_distance]),
    'average': widgets.VBox([linkage_distance]), 'ward': widgets.VBox([linkage_distance]),
    'groundtruth': widgets.VBox([ligand_file_control, ligand_distance]),
}
method_parameter_box = widgets.VBox()
def update_method_panel(change=None):
    method_parameter_box.children = (method_panels[method_control.value],)
method_control.observe(update_method_panel, names='value')
update_method_panel()

def current_method_parameters():
    method = method_control.value
    if method == 'meanshift': return {'automatic_bandwidth': meanshift_auto.value, 'quantile': meanshift_quantile.value, 'bandwidth': meanshift_bandwidth.value}
    if method == 'dbscan': return {'eps': dbscan_eps.value, 'min_samples': dbscan_min_samples.value}
    if method == 'louvain': return {'cutoff': louvain_cutoff.value, 'resolution': louvain_resolution.value}
    if method in ('single', 'complete', 'average', 'ward'): return {'distance_threshold': linkage_distance.value}
    return {'ligand_pdb_file': ligand_file_control.value, 'distance_threshold': ligand_distance.value}

run_button = widgets.Button(description='Run clustering', button_style='primary', icon='play')
output = widgets.Output()
def on_run_clicked(_):
    with output:
        clear_output(wait=True)
        try:
            pockets, residues = run_pocket_clustering(
                pdb_file=pdb_file_control.value, method=method_control.value, score_threshold=score_control.value,
                min_atom_count=min_atoms_control.value, score_type=score_type_control.value,
                output_prefix=prefix_control.value, **current_method_parameters())
            if len(pockets):
                display(Markdown('### Pocket summary'))
                display(pockets.style.format({'pocket_score': '{:.4f}'}))
                display(Markdown('### Residues by pocket'))
                for pocket_rank in pockets['pocket_rank']:
                    pocket_residues = residues.loc[residues['pocket_rank'] == pocket_rank, 'residue_id'].tolist()
                    display(Markdown(f'**Pocket {pocket_rank}:** ' + ', '.join(pocket_residues)))
        except Exception as exc:
            print(f'{type(exc).__name__}: {exc}')

run_button.on_click(on_run_clicked)
controls = widgets.VBox([pdb_file_control, method_control, method_parameter_box, score_control,
                         min_atoms_control, score_type_control, prefix_control, run_button])
display(controls, output)

Output()